# Lecture 06 — Clustering: K-means and PCA geometry

Synthetic customer summaries illustrate distance, local optimization, scaling, and visualization. Truth labels stay out of clustering and selection.

![Teaching figure for clustering](../../docs/figures/06_clustering.svg)

*Illustrative k=4 fit to all supplied customer features after standardization, projected to two principal components. This is not the later selected-k result.*

## What question does this clustering answer?

This is descriptive segmentation of the supplied customer population. We fit scaling and clustering on all observed rows because there is no held-out target to predict in this question. If the goal were reliable assignment of future customers, we would need an evaluation design for that future population and a preprocessing rule fitted before those future observations arrive.

The first four-cluster plot is an **initial illustration**, not the silhouette-selected final result. The later `chosen_k` fit is a separate model; do not interpret the initial plot as its output. Cluster IDs are nominal: cluster 0 is not smaller or better than cluster 1.

**One Lloyd update:** in one dimension, observations 0, 2, and 8 assigned to centers 0 and 7 form groups {0, 2} and {8}. Updating centers to group means gives 1 and 8. Reassign and repeat until the chosen stopping criterion is met. Standardizing inputs changes this geometry; it is a modeling choice, not just a numerical convenience. Two random starts illustrate sensitivity, not a complete stability study.

## 1. Alternating minimization

K-means minimizes $J=\sum_i\|z_i-\mu_{c_i}\|^2$. With centers fixed, nearest-center assignment minimizes each row's term. With assignments fixed, the gradient is $2|C_j|\mu_j-2\sum_{i\in C_j}z_i$, so the minimum is the arithmetic mean. These steps decrease objective, but the joint problem is nonconvex; initialization can find a local optimum. Multiple starts help without proving global optimality.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score


def repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "data" / "customer_segments.csv").exists():
            return path
    raise FileNotFoundError("Could not find data/customer_segments.csv")


ROOT = repo_root()
customers = pd.read_csv(ROOT / "data" / "customer_segments.csv")
features = [
    "recency_days",
    "orders_12m",
    "average_order_value",
    "annual_spend",
    "web_share",
    "tenure_months",
    "support_contacts_12m",
]
X = customers[features]
truth = customers["segment_truth"]  # Keep out of every fit and selection step.
display(X.describe().round(2))
scaler = StandardScaler()
Z = scaler.fit_transform(X)
print("means:", np.round(Z.mean(axis=0), 3), "stds:", np.round(Z.std(axis=0), 3))

### Scaling changes which differences count

Compare two observed pairs below. In raw units, a large currency-valued difference can dominate the sum of squared differences. Standardization divides by observed feature scales. Neither distance is intrinsically correct: each expresses a choice about comparable customer differences.

In [ ]:
distance_rows = []
for i, j in [(0, 1), (0, 10)]:
    distance_rows.append({"pair": f"{i}, {j}",
                          "raw_distance": np.linalg.norm(X.iloc[i] - X.iloc[j]),
                          "standardized_distance": np.linalg.norm(Z[i] - Z[j])})
display(pd.DataFrame(distance_rows))

## 2. Scaling defines distance

Large-scale raw features dominate Euclidean distance. Standardizing $z_{ij}=(x_{ij}-\bar x_j)/s_j$ makes one standard deviation the unit for every feature. This is a modeling choice about feature weights; skew and outliers remain.

In [ ]:
model = KMeans(
    n_clusters=4,
    init="k-means++",
    n_init=30,
    max_iter=300,
    random_state=606,
    algorithm="lloyd",
)
labels = model.fit_predict(Z)
print("inertia:", model.inertia_)
print("sizes:", np.bincount(labels))
centers = scaler.inverse_transform(model.cluster_centers_)
display(pd.DataFrame(centers, columns=features).round(2))

## 3. PCA from the Rayleigh quotient

The first PCA direction maximizes $v^TZ^TZv$ subject to $v^Tv=1$. A Lagrange multiplier gives $Z^TZv=\lambda v$, the leading eigenvector or right singular vector. The next code projects only for display. K-means fits all seven standardized original features; two dimensions can hide structure.

For centered $Z=U\Sigma V^T$, the columns of $V$ are principal directions. Scores in $r$ dimensions are $ZV_r=U_r\Sigma_r$, and the sample variance of component $j$ is $s_j^2/(n-1)$. Its explained-variance fraction is $s_j^2/\sum_l s_l^2$. The reconstruction $ZV_rV_r^T$ minimizes squared Frobenius reconstruction error among rank-$r$ matrices, with error $\sum_{j>r}s_j^2$. A component's sign is arbitrary; tied singular values allow rotations within the tied subspace. Neither sign flips nor rotations inside that subspace change the represented geometry.


In [ ]:
pca = PCA(n_components=2, svd_solver="full")
Z_2d = pca.fit_transform(Z)
print("explained variance fractions:", pca.explained_variance_ratio_.round(3))
points = plt.scatter(Z_2d[:, 0], Z_2d[:, 1], c=labels, cmap="tab10", s=16, alpha=0.75)
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("Initial illustrative k=4 fit, projected to PCA 2-D")
handles, _ = points.legend_elements()
plt.legend(handles, [f"Cluster {i}" for i in range(4)], title="Nominal IDs")
plt.tight_layout()

## 4. Internal diagnostics and label invariance

The globally minimal inertia cannot increase with $k$, because an added center can reproduce the previous solution. A finite randomized run can still wiggle across k when it reaches different local optima, so inspect the observed curve. An elbow remains heuristic. Silhouette compares within distance $a(i)$ to nearest other-cluster distance $b(i)$: $(b-a)/\max(a,b)$. It favors compact separated groups in this metric and can mislead for nonconvex or uneven-density patterns. Cluster IDs are arbitrary; adjusted Rand index (ARI) compares co-membership invariantly under label permutation. Use truth only after choices are fixed.

For a non-singleton cluster, $a(i)$ is the mean distance from point $i$ to the other points in its own cluster. For each other cluster, compute the mean distance from $i$ to its members; $b(i)$ is the smallest of those means. The silhouette is $(b(i)-a(i))/\max\{a(i),b(i)\}$ when the denominator is positive. Implementations assign silhouette zero to singleton clusters. This is a diagnostic of the chosen distance geometry, not external evidence that the partition is economically meaningful.


In [ ]:
rows, fitted = [], {}
for k in range(2, 9):
    estimator = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=20,
        max_iter=300,
        random_state=606,
        algorithm="lloyd",
    )
    labels = estimator.fit_predict(Z)
    fitted[k] = estimator
    rows.append(
        {
            "k": k,
            "inertia": estimator.inertia_,
            "silhouette": silhouette_score(Z, labels),
        }
    )
diagnostics = pd.DataFrame(rows).set_index("k")
display(diagnostics.round(3))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
diagnostics["inertia"].plot(marker="o", ax=axes[0])
axes[0].set(xlabel="k", ylabel="within-cluster sum of squares", title="Inertia")
diagnostics["silhouette"].plot(marker="o", ax=axes[1])
axes[1].set(xlabel="k", ylabel="mean silhouette", title="Internal diagnostic")
plt.tight_layout()

In [ ]:
chosen_k = int(diagnostics["silhouette"].idxmax())
chosen = fitted[chosen_k]
print("post-hoc synthetic ARI:", adjusted_rand_score(truth, chosen.labels_))
repeat = KMeans(
    n_clusters=chosen_k,
    n_init=1,
    init="random",
    random_state=17,
    max_iter=300,
    algorithm="lloyd",
).fit(Z)
print("ARI across starts:", adjusted_rand_score(chosen.labels_, repeat.labels_))
print("inertias:", chosen.inertia_, repeat.inertia_)

## 5. Interpret the selected solution, not the initial illustration

`chosen_k` maximizes silhouette among the candidate fits on the standardized seven-dimensional observations. Now show that exact model's sizes, profiles, and colors. The two-dimensional projection can overlap groups separated in other dimensions. Cluster IDs have no ordering and can permute between fits. The synthetic truth is used only for a post-hoc comparison, never for fitting or selecting k.

In [ ]:
print("Silhouette-selected k:", chosen_k)
selected_sizes = pd.Series(np.bincount(chosen.labels_, minlength=chosen_k), name="customers")
selected_profiles = pd.DataFrame(
    scaler.inverse_transform(chosen.cluster_centers_), columns=features
)
selected_profiles.index.name = "cluster"
display(selected_sizes.rename_axis("cluster").to_frame())
display(selected_profiles.round(2))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
colors = plt.get_cmap("tab10")
for cluster in range(chosen_k):
    mask = chosen.labels_ == cluster
    axes[0].scatter(Z_2d[mask, 0], Z_2d[mask, 1], s=12, alpha=.65,
                    color=colors(cluster), label=f"Cluster {cluster}")
axes[0].set(xlabel="PC 1", ylabel="PC 2", title=f"Selected k={chosen_k}: PCA display")
axes[0].legend(fontsize=8)
axes[1].bar(selected_sizes.index, selected_sizes, color=[colors(i) for i in selected_sizes.index])
axes[1].set(xlabel="Nominal cluster ID", ylabel="Customers", title="Selected cluster sizes")
axes[1].set_xticks(selected_sizes.index)
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(11, 3.5))
profile_image = ax.imshow(chosen.cluster_centers_, aspect="auto", cmap="RdBu_r")
ax.set_xticks(range(len(features)), features, rotation=30, ha="right")
ax.set_yticks(range(chosen_k), [f"Cluster {i}" for i in range(chosen_k)])
ax.set_title("Selected centroids in standardized feature units")
fig.colorbar(profile_image, ax=ax, label="Standard deviations from observed feature mean")
plt.tight_layout()
plt.show()

### Sensitivity across several random starts

Hold k, the dataset, and preprocessing fixed. Ten single-start fits measure initialization sensitivity on these observations. This does not measure stability under new customers or resampling, and agreement with the selected multi-start fit is not a ground-truth accuracy score.

In [ ]:
stability_rows = []
for seed in range(10):
    trial = KMeans(n_clusters=chosen_k, n_init=1, init="random", random_state=seed).fit(Z)
    stability_rows.append({"seed": seed, "inertia": trial.inertia_,
                           "ARI_vs_selected": adjusted_rand_score(chosen.labels_, trial.labels_)})
stability_table = pd.DataFrame(stability_rows)
display(stability_table.round(4))
fig, ax = plt.subplots(figsize=(8, 3))
ax.scatter(stability_table.seed, stability_table.ARI_vs_selected)
ax.set(xlabel="Random seed", ylabel="ARI versus selected fit", ylim=(-.05, 1.05),
       title="Initialization sensitivity: ten single-start fits")
plt.tight_layout()
plt.show()

## Limits

K-means favors roughly spherical groups under Euclidean distance and can be sensitive to outliers, scaling, and initialization. A partition is not proof of natural or actionable customer types.